In [112]:
import numpy as np
import pandas as pd
from pathlib import Path

# All Formula 1 CSVs (Ergast / Kaggle "F1 World Championship 1950-2020" schema)
# This notebook lives in the same folder as the CSVs, so load from the current dir.
DATA_DIR = Path(".")

dfs = {p.stem: pd.read_csv(p) for p in sorted(DATA_DIR.glob("*.csv"))}

# Expose each table as its own variable: circuits, drivers, races, results, ...
globals().update(dfs)

for name, df in dfs.items():
    print(f"{name:<22} {df.shape[0]:>7} rows x {df.shape[1]:>2} cols")

Formula1results          27392 rows x 18 cols
circuits                    78 rows x  9 cols
constructor_results      12942 rows x  5 cols
constructor_standings    13708 rows x  7 cols
constructors               214 rows x  5 cols
driver_standings         35515 rows x  7 cols
drivers                    865 rows x  9 cols
lap_times               873755 rows x  6 cols
pit_stops                22383 rows x  7 cols
qualifying               11124 rows x  9 cols
races                     1171 rows x 18 cols
results                  27392 rows x 18 cols
seasons                     77 rows x  2 cols
sprint_results             546 rows x 17 cols
status                     140 rows x  2 cols


In [113]:
# Read a single file into a pandas DataFrame
df = pd.read_csv("results.csv")

# 1) How many rows and columns? -> (rows, columns)
print("Shape (rows, columns):", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Shape (rows, columns): (27392, 18)
Number of rows: 27392
Number of columns: 18


In [114]:
# 2) What columns are there?
print("Columns:")
print(list(df.columns))

Columns:
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']


In [115]:
# 3) What data type is each column? (object = text/string in pandas)
print(df.dtypes)

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number              object
grid                object
position            object
positionText        object
positionOrder        int64
points             float64
laps                 int64
time                object
milliseconds        object
fastestLap          object
rank                object
fastestLapTime      object
fastestLapSpeed     object
statusId             int64
dtype: object


In [116]:
# 4) Everything at once: rows, columns, dtypes, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27392 entries, 0 to 27391
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         27392 non-null  int64  
 1   raceId           27392 non-null  int64  
 2   driverId         27392 non-null  int64  
 3   constructorId    27392 non-null  int64  
 4   number           27392 non-null  object 
 5   grid             27392 non-null  object 
 6   position         27392 non-null  object 
 7   positionText     27392 non-null  object 
 8   positionOrder    27392 non-null  int64  
 9   points           27392 non-null  float64
 10  laps             27392 non-null  int64  
 11  time             27392 non-null  object 
 12  milliseconds     27392 non-null  object 
 13  fastestLap       27392 non-null  object 
 14  rank             27392 non-null  object 
 15  fastestLapTime   27392 non-null  object 
 16  fastestLapSpeed  27392 non-null  object 
 17  statusId    

In [117]:
# 5) Peek at the first 5 rows to see what the data actually looks like
df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


## 6) Merging tables — show driver *names* and *country*, not just `driverId`

`results` stores a numeric `driverId` instead of a name (to avoid repeating
"Lewis Hamilton" thousands of times). The `drivers` table is the lookup that
maps each `driverId` -> `forename`, `surname`, `nationality`, etc.

**Merging** = joining two tables on a shared key column. Here the key is
`driverId`, which exists in both tables.

- `on="driverId"` — the shared column to match rows by
- `how="left"`   — keep every row from the left table (`results`) and attach
  driver info where it matches (so we never lose a race result)

**Where does the country come from?** The `drivers` table already has a
`nationality` column (all 865 drivers filled in), so no external lookup is
needed. `nationality` is a demonym ("British"), so we map it to a real country
name ("United Kingdom") to create a clean `country` column.

In [118]:
# Grab the columns we need from the drivers lookup table.
# The drivers table ALREADY has a "nationality" column, so we don't need to
# search anywhere external for which country a driver is from.
driver_info = drivers[["driverId", "forename", "surname", "nationality"]]

# Join driver info onto every result row, matching on driverId
merged = results.merge(driver_info, on="driverId", how="left")

# Combine forename + surname into a single readable "driver" column
merged["driver"] = merged["forename"] + " " + merged["surname"]

# "nationality" is a demonym (British, Dutch, ...). Map it to an actual country
# name so we have a clean "country" column.
demonym_to_country = {
    "American": "United States", "American-Italian": "United States",
    "Argentine": "Argentina", "Argentine-Italian": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "United Kingdom", "Canadian": "Canada",
    "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France",
    "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Indonesian": "Indonesia", "Irish": "Ireland", "Italian": "Italy",
    "Japanese": "Japan", "Liechtensteiner": "Liechtenstein", "Malaysian": "Malaysia",
    "Mexican": "Mexico", "Monegasque": "Monaco", "New Zealander": "New Zealand",
    "Polish": "Poland", "Portuguese": "Portugal", "Rhodesian": "Zimbabwe",
    "Russian": "Russia", "South African": "South Africa", "Spanish": "Spain",
    "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela",
}
# .str.strip() handles a stray trailing space in the data ("Argentinian ")
merged["country"] = merged["nationality"].str.strip().map(demonym_to_country)

# Show results with the driver's name, nationality and country next to the IDs.
# Reminder: column selection uses TWO bracket pairs -> merged[[ "a", "b" ]]
merged[["driverId", "driver", "nationality", "country",
        "constructorId", "grid", "position", "points"]].head()

,driverId,driver,nationality,country,constructorId,grid,position,points
0,1,Lewis Hamilton,British,United Kingdom,1,1,1,10.0
1,2,Nick Heidfeld,German,Germany,2,5,2,8.0
2,3,Nico Rosberg,German,Germany,3,7,3,6.0
3,4,Fernando Alonso,Spanish,Spain,4,11,4,5.0
4,5,Heikki Kovalainen,Finnish,Finland,1,3,5,4.0


In [119]:
# Same pattern, chained: each extra table is just another .merge() on its ID
out = (
    results
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]], on="constructorId", how="left")
    .merge(races[["raceId", "name", "year"]], on="raceId", how="left",
           suffixes=("_constructor", "_race"))  # both tables have a "name" column
)

out["driver"] = out["forename"] + " " + out["surname"]
out = out.rename(columns={"name_constructor": "team", "name_race": "grand_prix"})

out[["year", "grand_prix", "driver", "team", "grid", "position", "points"]].head()

,year,grand_prix,driver,team,grid,position,points
0,2008,Australian Grand Prix,Lewis Hamilton,McLaren,1,1,10.0
1,2008,Australian Grand Prix,Nick Heidfeld,BMW Sauber,5,2,8.0
2,2008,Australian Grand Prix,Nico Rosberg,Williams,7,3,6.0
3,2008,Australian Grand Prix,Fernando Alonso,Renault,11,4,5.0
4,2008,Australian Grand Prix,Heikki Kovalainen,McLaren,3,5,4.0


## 7) `Formula1results` — `results`, but with names instead of IDs

A clean copy of the `results` table where every **ID** foreign-key column is
replaced by the real value it points to:

| results column | becomes | from lookup table |
|---|---|---|
| `raceId`        | `race` (Grand Prix name) | `races` |
| `driverId`      | `driver` (forename + surname) | `drivers` |
| `constructorId` | `constructor` (team name) | `constructors` |
| `statusId`      | `status` (e.g. "Finished") | `status` |

`resultId` is kept — it's the table's own primary key (the unique id of each
result row), not a reference to another table, so there's no name to swap in.
`number` is the driver's car number, not a foreign key, so it stays too.

In [120]:
# Build small lookup tables, renaming each "name" column so they don't collide
races_lk        = races[["raceId", "name"]].rename(columns={"name": "race"})
constructors_lk = constructors[["constructorId", "name"]].rename(columns={"name": "constructor"})
status_lk       = status[["statusId", "status"]]

# Join every lookup onto results, then build the driver's full name
f1 = (
    results
    .merge(races_lk,                                  on="raceId",        how="left")
    .merge(drivers[["driverId", "forename", "surname"]], on="driverId",   how="left")
    .merge(constructors_lk,                           on="constructorId", how="left")
    .merge(status_lk,                                 on="statusId",      how="left")
)
f1["driver"] = f1["forename"] + " " + f1["surname"]

# Same column order as `results`, with each ID column swapped for its real value
Formula1results = f1[[
    "resultId", "race", "driver", "constructor", "number", "grid",
    "position", "positionText", "positionOrder", "points", "laps", "time",
    "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed",
    "status",
]]

# Save it next to the other CSVs so it's a real dataset, then preview
Formula1results.to_csv("Formula1results.csv", index=False)
print("Formula1results:", Formula1results.shape, "-> saved to Formula1results.csv")
Formula1results.head()

Formula1results: (27392, 18) -> saved to Formula1results.csv


,resultId,race,driver,constructor,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,status
0,1,Australian Grand Prix,Lewis Hamilton,McLaren,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.3,Finished
1,2,Australian Grand Prix,Nick Heidfeld,BMW Sauber,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,Finished
2,3,Australian Grand Prix,Nico Rosberg,Williams,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,Finished
3,4,Australian Grand Prix,Fernando Alonso,Renault,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,Finished
4,5,Australian Grand Prix,Heikki Kovalainen,McLaren,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,Finished


## 8) Quick questions — most wins, podiums, and points

All three use the readable `Formula1results` table (driver names instead of IDs).

- **`position`** is text and stores `\N` for cars that didn't finish, so we compare
  it as a string: a win is `position == "1"`, a podium is `position` in 1/2/3.
- **Points** are summed per driver with `groupby`.

In [121]:
# Q1: Which driver has the most wins? (finished in position 1)
wins = Formula1results[Formula1results["position"] == "1"]["driver"].value_counts()
print("Most wins:", wins.index[0], "-", int(wins.iloc[0]))
wins.head(10)

Most wins: Lewis Hamilton - 106


driver
Lewis Hamilton        106
Michael Schumacher     91
Max Verstappen         71
Sebastian Vettel       53
Alain Prost            51
Ayrton Senna           41
Fernando Alonso        32
Nigel Mansell          31
Jackie Stewart         27
Jim Clark              25
Name: count, dtype: int64

In [122]:
# Q2: Which driver has the most podiums? (finished 1st, 2nd or 3rd)
podiums = Formula1results[Formula1results["position"].isin(["1", "2", "3"])]["driver"].value_counts()
print("Most podiums:", podiums.index[0], "-", int(podiums.iloc[0]))
podiums.head(10)

Most podiums: Lewis Hamilton - 206


driver
Lewis Hamilton        206
Michael Schumacher    155
Max Verstappen        128
Sebastian Vettel      122
Alain Prost           106
Fernando Alonso       106
Kimi Räikkönen        103
Ayrton Senna           80
Rubens Barrichello     68
Valtteri Bottas        67
Name: count, dtype: int64

In [123]:
# Q3: Who has the most total points? (summed across every race)
points = Formula1results.groupby("driver")["points"].sum().sort_values(ascending=False)
print("Most points:", points.index[0], "-", points.iloc[0])
points.head(10)

Most points: Lewis Hamilton - 5057.5


driver
Lewis Hamilton        5057.5
Max Verstappen        3350.5
Sebastian Vettel      3098.0
Fernando Alonso       2381.0
Kimi Räikkönen        1873.0
Valtteri Bottas       1788.0
Charles Leclerc       1650.0
Nico Rosberg          1594.5
Sergio Pérez          1585.0
Michael Schumacher    1566.0
Name: points, dtype: float64

A few columns are stored as **text** with `\N` for missing values, so we first
make clean numeric helper columns:

- `grid_n`   — starting position (`grid == 0` means a pit-lane start, treated as missing)
- `finish_n` — finishing position (missing = did not finish / not classified)
- `rank_n`   — fastest-lap rank (`rank_n == 1` means that driver set the race's fastest lap)

"Best average" uses a **minimum number of entries** so a driver with one lucky
race doesn't top the list.

In [124]:
# --- Setup: numeric helper columns + a country column on Formula1results ---
fr = Formula1results.copy()
fr["grid_n"]   = pd.to_numeric(fr["grid"], errors="coerce").replace(0, np.nan)
fr["finish_n"] = pd.to_numeric(fr["position"], errors="coerce")
fr["rank_n"]   = pd.to_numeric(fr["rank"], errors="coerce")

# attach country (reuse the demonym_to_country map from section 6)
dn = drivers.copy()
dn["driver"]  = dn["forename"] + " " + dn["surname"]
dn["country"] = dn["nationality"].str.strip().map(demonym_to_country)
fr = fr.merge(dn[["driver", "country"]], on="driver", how="left")

print("helper columns ready:", [c for c in fr.columns if c.endswith("_n")] + ["country"])

helper columns ready: ['grid_n', 'finish_n', 'rank_n', 'country']


In [125]:
# Q1: Best average start & finish position (lower number = better).
# Only drivers with >= 50 entries, so the averages are meaningful.
MIN_ENTRIES = 50
g = fr.groupby("driver")
avg = pd.DataFrame({
    "entries":    g.size(),
    "avg_start":  g["grid_n"].mean().round(2),
    "avg_finish": g["finish_n"].mean().round(2),
})
avg = avg[avg["entries"] >= MIN_ENTRIES]

print("Best AVERAGE START position:")
print(avg.sort_values("avg_start").head(10)[["entries", "avg_start"]], "\n")
print("Best AVERAGE FINISH position:")
print(avg.sort_values("avg_finish").head(10)[["entries", "avg_finish"]])

Best AVERAGE START position:
                    entries  avg_start
driver                                
Juan Fangio              58       2.48
Ayrton Senna            162       3.15
Jim Clark                73       3.60
Alain Prost             202       4.16
Jackie Stewart          100       4.50
Lewis Hamilton          387       4.60
Michael Schumacher      308       4.87
Max Verstappen          240       4.93
Stirling Moss            73       5.18
Juan Pablo Montoya       95       5.37 

Best AVERAGE FINISH position:
                    entries  avg_finish
driver                                 
Juan Fangio              58        2.25
Jackie Stewart          100        2.95
Alain Prost             202        2.96
Ayrton Senna            162        3.15
Stirling Moss            73        3.46
Michael Schumacher      308        3.70
Max Verstappen          240        3.75
Jim Clark                73        3.84
Nigel Mansell           192        3.92
Lewis Hamilton          387    

In [126]:
# Q2: Which country has the best / worst drivers?
# Metric = average points scored per race entry (only countries with >= 200 entries).
# NOTE: F1's points system changed a lot over the decades (more points awarded in
# modern eras), so this leans toward countries with strong *recent* drivers.
c = fr.groupby("country")
country = pd.DataFrame({
    "entries":    c.size(),
    "avg_points": c["points"].mean().round(3),
    "wins":       fr[fr["position"] == "1"].groupby("country").size(),
}).fillna({"wins": 0})
country = country[country["entries"] >= 200].sort_values("avg_points", ascending=False)

print("BEST countries (by avg points per entry):")
print(country.head(5), "\n")
print("WORST countries:")
print(country.tail(5))

BEST countries (by avg points per entry):
             entries  avg_points  wins
country                               
Monaco           209       7.914   8.0
Netherlands      567       5.960  71.0
Australia        930       3.895  52.0
Spain            975       3.822  36.0
Finland         1200       3.657  57.0 

WORST countries:
               entries  avg_points  wins
country                                 
United States     1317       0.759  33.0
Switzerland        496       0.702   7.0
Italy             3449       0.672  48.0
Belgium            591       0.657  11.0
Japan              705       0.450   0.0


In [127]:
# Q2B: Which country has the best / worst drivers?
# Metric = average points scored per race entry (only countries with >= 200 entries).
# NOTE: F1's points system changed a lot over the decades (more points awarded in
# modern eras), so this leans toward countries with strong *recent* drivers.
c = fr.groupby("country")
country = pd.DataFrame({
    "entries":    c.size(),
    "avg_points": c["points"].mean().round(3),
    "wins":       fr[fr["position"] == "1"].groupby("country").size(),
}).fillna({"wins": 0})
country = country[country["entries"] >= 200].sort_values("wins", ascending=False)

print("BEST countries (by wins):")
print(country.head(5), "\n")
print("WORST countries:")
print(country.tail(5))

BEST countries (by wins):
                entries  avg_points   wins
country                                   
United Kingdom     4690       2.779  328.0
Germany            2466       3.260  179.0
Brazil             1984       1.736  101.0
France             3188       1.197   81.0
Netherlands         567       5.960   71.0 

WORST countries:
             entries  avg_points  wins
country                               
Monaco           209       7.914   8.0
Switzerland      496       0.702   7.0
Denmark          221       0.891   0.0
Russia           213       1.254   0.0
Japan            705       0.450   0.0


In [128]:
# Q3: Who has set the most fastest laps? (fastest lap of a race == rank_n 1)
fastest_laps = fr[fr["rank_n"] == 1]["driver"].value_counts()
print("Most fastest laps:", fastest_laps.index[0], "-", int(fastest_laps.iloc[0]))
fastest_laps.head(10)

Most fastest laps: Lewis Hamilton - 68


driver
Lewis Hamilton        68
Kimi Räikkönen        42
Sebastian Vettel      38
Max Verstappen        37
Fernando Alonso       25
Michael Schumacher    21
Nico Rosberg          20
Valtteri Bottas       19
Lando Norris          19
Mark Webber           19
Name: count, dtype: int64

In [129]:
# Q4: Do drivers who start higher usually finish higher?
# Correlate starting position (grid_n) with finishing position (finish_n).
# Both use "1 = best", so a POSITIVE correlation means start higher -> finish higher.
pair = fr[["grid_n", "finish_n"]].dropna()
corr = pair["grid_n"].corr(pair["finish_n"])
print(f"Correlation(start, finish) = {corr:.3f}")


Correlation(start, finish) = 0.644


In [130]:
# Q5: Who completed the most laps overall? (sum of laps across every race)
total_laps = fr.groupby("driver")["laps"].sum().sort_values(ascending=False)
print("Most laps completed:", total_laps.index[0], "-", int(total_laps.iloc[0]))
total_laps.head(10)

Most laps completed: Fernando Alonso - 23383


driver
Fernando Alonso       23383
Lewis Hamilton        22034
Kimi Räikkönen        18618
Michael Schumacher    16824
Rubens Barrichello    16642
Sebastian Vettel      16426
Jenson Button         16274
Sergio Pérez          15890
Felipe Massa          14852
Valtteri Bottas       14130
Name: laps, dtype: int64

## 10) Start-vs-finish, overall ranking, pole conversion & pit stops

This section builds one analysis frame `fa` that keeps the **IDs** (so we can
join qualifying / pit-stop tables) *and* the **names**, plus numeric helpers:

- `gain   = grid_n - finish_n` → **+ve means gained places** vs the start
- `won    = position == "1"`, `podium = finish in top 3`
- `n_stops` → number of pit stops that race (from `pit_stops`)

⚠️ **Important caveat for "positions gained/lost":** how many places you can
gain is bounded by where you start. Back-of-grid cars rack up big gains as
others retire; front-row cars can only stay put or lose. So raw gain/loss
mostly reflects *grid position*, not driver skill — read those two tables with
that in mind.

In [131]:
# --- Section 10 setup: analysis frame with IDs + names + helper columns ---
dn = drivers.copy()
dn["driver"]  = dn["forename"] + " " + dn["surname"]
dn["country"] = dn["nationality"].str.strip().map(demonym_to_country)

fa = (
    results
    .merge(dn[["driverId", "driver", "country"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
    .merge(races[["raceId", "name"]].rename(columns={"name": "race"}), on="raceId", how="left")
)
fa["grid_n"]   = pd.to_numeric(fa["grid"], errors="coerce").replace(0, np.nan)
fa["finish_n"] = pd.to_numeric(fa["position"], errors="coerce")
fa["gain"]     = fa["grid_n"] - fa["finish_n"]      # +ve = gained places vs start
fa["won"]      = fa["position"].eq("1")
fa["podium"]   = fa["finish_n"].isin([1, 2, 3])

# number of pit stops per (race, driver) — the max "stop" number that race
n_stops = pit_stops.groupby(["raceId", "driverId"])["stop"].max().rename("n_stops")
fa = fa.merge(n_stops, on=["raceId", "driverId"], how="left")

print("analysis frame `fa`:", fa.shape)

analysis frame `fa`: (27392, 28)


In [132]:
# Q: Who usually GAINS the most positions from start to finish?
MIN = 50
g = fa.groupby("driver")
gain = pd.DataFrame({
    "entries":   g.size(),
    "avg_start": g["grid_n"].mean().round(1),
    "avg_gain":  g["gain"].mean().round(2),   # +ve = climbs forward on average
})
gain = gain[gain["entries"] >= MIN]
print("Most positions GAINED start -> finish (avg per race):")
print(gain.sort_values("avg_gain", ascending=False).head(20))
print("\nNotice the 'avg_start' column: these are all back-of-grid drivers (start"
      "\n~P20) who climb as cars ahead retire. Gain is capped by where you start.")

Most positions GAINED start -> finish (avg per race):
                    entries  avg_start  avg_gain
driver                                          
Piercarlo Ghinzani      111       22.0     11.15
Gabriele Tarquini        78       20.8     10.85
Jonathan Palmer          88       20.6     10.82
Yannick Dalmas           50       21.8     10.13
Hector Rebaque           58       17.3     10.00
Olivier Grouillard       62       19.3      9.64
Philippe Streiff         54       18.2      9.47
Érik Comas               63       18.1      9.00
Jackie Oliver            51       14.0      8.62
Nicola Larini            75       18.6      8.54
Alex Caffi               75       19.0      8.50
Luca Badoer              58       20.3      8.40
Marc Surer               88       17.4      8.27
Philippe Alliot         115       18.1      7.77
Derek Daly               64       17.0      7.69
Pedro Diniz              99       17.2      7.64
Ukyo Katayama            97       17.3      7.27
Jyrki Järvileht

In [133]:
# Q: Overall drivers ranking + win rate
g = fa.groupby("driver")
rank = pd.DataFrame({
    "entries":    g.size(),
    "wins":       fa[fa["won"]].groupby("driver").size(),
    "podiums":    fa[fa["podium"]].groupby("driver").size(),
    "points":     g["points"].sum(),
    "avg_finish": g["finish_n"].mean().round(2),
}).fillna(0)
rank["win_rate"] = (rank["wins"] / rank["entries"]).round(3)

print("OVERALL RANKING (by total career points):")
print(rank.sort_values("points", ascending=False).head(10)[["entries", "wins", "podiums", "points", "win_rate"]])

print("\nHighest WIN RATE (wins / entries, min 50 entries):")
print(rank[rank["entries"] >= 50].sort_values("win_rate", ascending=False)
          .head(10)[["entries", "wins", "win_rate"]])

OVERALL RANKING (by total career points):
                    entries   wins  podiums  points  win_rate
driver                                                       
Lewis Hamilton          387  106.0    206.0  5057.5     0.274
Max Verstappen          240   71.0    128.0  3350.5     0.296
Sebastian Vettel        300   53.0    122.0  3098.0     0.177
Fernando Alonso         435   32.0    106.0  2381.0     0.074
Kimi Räikkönen          352   21.0    103.0  1873.0     0.060
Valtteri Bottas         254   10.0     67.0  1788.0     0.039
Charles Leclerc         180    8.0     52.0  1650.0     0.044
Nico Rosberg            206   23.0     57.0  1594.5     0.112
Sergio Pérez            290    6.0     39.0  1585.0     0.021
Michael Schumacher      308   91.0    155.0  1566.0     0.295

Highest WIN RATE (wins / entries, min 50 entries):
                    entries   wins  win_rate
driver                                      
Juan Fangio              58   24.0     0.414
Jim Clark                73

In [134]:
# Q: Who qualifies high but finishes lower? Does it depend on team / track?
# Look only at GOOD qualifiers (avg start <= 8) so we aren't just listing backmarkers.
g = fa.groupby("driver")
und = pd.DataFrame({
    "entries":         g.size(),
    "avg_start":       g["grid_n"].mean().round(1),
    "avg_places_lost": (-g["gain"].mean()).round(2),   # +ve = drops back vs start
})
und = und[(und["entries"] >= 50) & (und["avg_start"] <= 8)]
print("Good qualifiers who drop back the most on average:")
print(und.sort_values("avg_places_lost", ascending=False).head(8))

# Dependence on TEAM and TRACK (avg places lost; remember the grid-position caveat)
team_lost = (-fa.groupby("constructor")["gain"].mean())
team_lost = team_lost[fa.groupby("constructor").size() >= 200].sort_values(ascending=False).round(2)
print("\nMost places lost by TEAM (min 200 entries) — note strong teams start at the"
      "\nfront so naturally have little room to gain:")
print(team_lost.head(5))

track_lost = (-fa.groupby("race")["gain"].mean())
track_lost = track_lost[fa.groupby("race").size() >= 150].sort_values(ascending=False).round(2)
print("\nMost places lost by TRACK (min 150 entries):")
print(track_lost.head(15))

Good qualifiers who drop back the most on average:
                 entries  avg_start  avg_places_lost
driver                                              
Oscar Piastri         77        5.9             0.73
Lando Norris         159        6.5             0.42
Ayrton Senna         162        3.1             0.30
Jim Clark             73        3.6             0.14
Juan Fangio           58        2.5            -0.00
Charles Leclerc      180        6.3            -0.27
Nico Rosberg         206        6.9            -0.32
Lewis Hamilton       387        4.6            -0.68

Most places lost by TEAM (min 200 entries) — note strong teams start at the
front so naturally have little room to gain:
constructor
Mercedes         -0.44
Haas F1 Team     -0.71
Alpine F1 Team   -0.91
Red Bull         -1.02
Williams         -1.19
Name: gain, dtype: float64

Most places lost by TRACK (min 150 entries):
race
Turkish Grand Prix      -0.76
Bahrain Grand Prix      -0.97
Abu Dhabi Grand Prix    -1.00
Ch

In [135]:
# Q: How much does starting position predict a podium?
byslot = fa.dropna(subset=["grid_n"]).copy()
byslot["slot"] = byslot["grid_n"].round().astype(int)
podium_pct = (byslot.groupby("slot")["podium"].mean() * 100).round(1)

print("Podium % by starting slot (P1..P10):")
print(podium_pct.head(10))
print(f"\nPole (P1) -> podium {podium_pct.get(1)}%   vs   P10 -> podium {podium_pct.get(10)}%")
corr = byslot["grid_n"].corr(byslot["podium"].astype(int))
print(f"corr(grid, podium) = {corr:.3f}  (negative = lower grid number -> more podiums)")
print("So starting position predicts podiums strongly, but it's a steep curve:"
      "\nfront row ~60%, mid-grid single digits.")

Podium % by starting slot (P1..P10):
slot
1     64.0
2     56.3
3     46.5
4     34.3
5     24.8
6     18.3
7     12.8
8      9.7
9      8.2
10     5.7
Name: podium, dtype: float64

Pole (P1) -> podium 64.0%   vs   P10 -> podium 5.7%
corr(grid, podium) = -0.442  (negative = lower grid number -> more podiums)
So starting position predicts podiums strongly, but it's a steep curve:
front row ~60%, mid-grid single digits.


In [136]:
# Q: Does pole (start P1) always = win? Track dependence? Pit stops?
pole = fa[fa["grid_n"] == 1]
print(f"Pole -> win overall: {pole['won'].mean()*100:.1f}%  ->  NO, pole is far from a guaranteed win.\n")

# TRACK dependence: hard-to-overtake circuits convert pole into wins much more often
pt = pole.groupby("race")["won"].agg(["mean", "size"])
pt = pt[pt["size"] >= 15].sort_values("mean", ascending=False)
print("Easiest tracks to convert pole -> win (min 15 poles):")
print((pt["mean"] * 100).round(0).head(5))
print("\nHardest tracks to convert pole -> win:")
print((pt["mean"] * 100).round(0).tail(5))

# PIT STOPS: do winners stop fewer times?  (pit-stop data only covers ~2011+)
stops = fa.dropna(subset=["n_stops"])
print("\nAvg number of pit stops — winners vs everyone else:")
print(stops.groupby("won")["n_stops"].mean().round(2))
print("=> almost identical, so 'number of stops' barely separates winners from the rest.")

Pole -> win overall: 43.1%  ->  NO, pole is far from a guaranteed win.

Easiest tracks to convert pole -> win (min 15 poles):
race
Abu Dhabi Grand Prix    71.0
Singapore Grand Prix    69.0
Chinese Grand Prix      63.0
Spanish Grand Prix      56.0
Japanese Grand Prix     55.0
Name: mean, dtype: float64

Hardest tracks to convert pole -> win:
race
Austrian Grand Prix         34.0
Italian Grand Prix          33.0
Argentine Grand Prix        32.0
European Grand Prix         30.0
South African Grand Prix    26.0
Name: mean, dtype: float64

Avg number of pit stops — winners vs everyone else:
won
False    1.97
True     1.93
Name: n_stops, dtype: float64
=> almost identical, so 'number of stops' barely separates winners from the rest.


## 11) Chaos, net movement, driver age & a head-to-head

We extend `fa` with three more columns, plus driver **age at each race**:

- `net   = grid_n - positionOrder` → net places moved (**+ve = forward**).
  Uses `positionOrder` (always present, 1..N) instead of `finish_n`, so a
  retirement counts as dropping to the back rather than being ignored.
- `chaos = |grid_n - positionOrder|` → how much a driver's start and finish
  *differ*, regardless of direction (your "chaotic races" metric).
- `age_at_race` = race `date` − driver `dob`, in years (joins `dob` from
  `drivers` and `date` from `races`).

Same caveat as section 10: net movement is bounded by where you start, so it
partly reflects grid position rather than pure skill.

In [137]:
# --- Section 11 setup: extend fa with net, chaos, and age_at_race ---
# add dob (from drivers) and the race date (from races)
fa = fa.merge(drivers[["driverId", "dob"]], on="driverId", how="left")
fa = fa.merge(races[["raceId", "date"]], on="raceId", how="left")

fa["net"]   = fa["grid_n"] - fa["positionOrder"]          # +ve = moved forward
fa["chaos"] = (fa["grid_n"] - fa["positionOrder"]).abs()  # magnitude of the swing
fa["age_at_race"] = (
    (pd.to_datetime(fa["date"], errors="coerce") - pd.to_datetime(fa["dob"], errors="coerce"))
    .dt.days / 365.25
)
print("added: net, chaos, age_at_race")
print("age range in data: %.1f - %.1f years" % (fa["age_at_race"].min(), fa["age_at_race"].max()))

added: net, chaos, age_at_race
age range in data: 17.5 - 58.8 years


In [138]:
# Q: Which driver has the most CHAOTIC races?  avg |grid - positionOrder|
MIN = 50
g = fa.groupby("driver")
chaos = pd.DataFrame({
    "entries":   g.size(),
    "avg_start": g["grid_n"].mean().round(1),
    "avg_chaos": g["chaos"].mean().round(2),   # bigger = start & finish differ a lot
})
chaos = chaos[chaos["entries"] >= MIN]
print("Most CHAOTIC drivers (avg |grid - positionOrder|):")
print(chaos.sort_values("avg_chaos", ascending=False).head(10))

Most CHAOTIC drivers (avg |grid - positionOrder|):
                       entries  avg_start  avg_chaos
driver                                              
Jean-Pierre Jabouille       55        9.5       8.86
Hector Rebaque              58       17.3       8.00
Marc Surer                  88       17.4       7.83
Philippe Streiff            54       18.2       7.81
Jonathan Palmer             88       20.6       7.79
Vittorio Brambilla          78       13.3       7.68
Yannick Dalmas              50       21.8       7.62
Érik Comas                  63       18.1       7.54
Keke Rosberg               128       10.1       7.41
Nicola Larini               75       18.6       7.41


In [139]:
# Q: Which drivers GAIN the most / LOSE the most positions (net, grid -> finish)?
MIN = 50
g = fa.groupby("driver")
mv = pd.DataFrame({
    "entries":   g.size(),
    "avg_start": g["grid_n"].mean().round(1),
    "avg_net":   g["net"].mean().round(2),     # +ve = moves up, -ve = drops back
})
mv = mv[mv["entries"] >= MIN]

print("GAIN the most positions (avg net forward):")
print(mv.sort_values("avg_net", ascending=False).head(10))

# Start HIGH but lose the most: restrict to good qualifiers (avg start <= 8)
front = mv[mv["avg_start"] <= 8]
print("\nStart HIGH (avg start <= 8) but LOSE the most positions:")
print(front.sort_values("avg_net").head(10))
print("\n(Remember: with `net` a retirement drops you to the back, so front-runners"
      "\nwho DNF often show big losses — it mixes pace with reliability.)")

GAIN the most positions (avg net forward):
                   entries  avg_start  avg_net
driver                                        
Yannick Dalmas          50       21.8     7.04
Jonathan Palmer         88       20.6     6.14
Luca Badoer             58       20.3     5.43
Marc Surer              88       17.4     5.10
Philippe Streiff        54       18.2     4.48
Rolf Stommelen          61       17.4     4.26
Érik Comas              63       18.1     4.25
Gabriele Tarquini       78       20.8     4.18
Derek Daly              64       17.0     4.00
Alex Caffi              75       19.0     3.96

Start HIGH (avg start <= 8) but LOSE the most positions:
                    entries  avg_start  avg_net
driver                                         
Ayrton Senna            162        3.1    -4.99
Gerhard Berger          210        6.4    -4.54
Nigel Mansell           192        6.4    -4.48
James Hunt               93        7.0    -4.39
Stirling Moss            73        5.2    -3.89

In [140]:
# Q: Do OLDER drivers gain more positions (or lose fewer) than younger drivers?
pair = fa[["age_at_race", "net"]].dropna()
print(f"corr(age_at_race, net) = {pair['age_at_race'].corr(pair['net']):.3f}  "
      "(~0 = basically no overall link)\n")

# Average net movement by age bucket
buckets = pd.cut(fa["age_at_race"], [0, 25, 30, 35, 60],
                 labels=["<25", "25-30", "30-35", "35+"])
by_age = fa.groupby(buckets, observed=True)["net"].agg(["mean", "size"]).round(3)
print("Average NET positions gained, by age bucket:")
print(by_age)
print("\nThe youngest group gains slightly more, but the effect is tiny and confounded:"
      "\nyoung drivers often sit in weaker cars starting further back, so they simply"
      "\nhave more room to climb. Age itself barely moves the needle.")

corr(age_at_race, net) = -0.022  (~0 = basically no overall link)

Average NET positions gained, by age bucket:
              mean   size
age_at_race              
<25          0.556   4865
25-30        0.006  10141
30-35       -0.284   7707
35+         -0.114   4679

The youngest group gains slightly more, but the effect is tiny and confounded:
young drivers often sit in weaker cars starting further back, so they simply
have more room to climb. Age itself barely moves the needle.


In [141]:
# Q: Lewis Hamilton vs Max Verstappen — head to head
def career(name):
    d = fa[fa["driver"] == name]
    return pd.Series({
        "entries":      len(d),
        "wins":         int((d["position"] == "1").sum()),
        "podiums":      int(d["finish_n"].isin([1, 2, 3]).sum()),
        "poles":        int((d["grid_n"] == 1).sum()),
        "fastest_laps": int((pd.to_numeric(d["rank"], errors="coerce") == 1).sum()),
        "points":       d["points"].sum(),
        "win_rate":     round((d["position"] == "1").mean(), 3),
        "avg_start":    round(d["grid_n"].mean(), 2),
        "avg_finish":   round(d["finish_n"].mean(), 2),
    })

h2h = pd.DataFrame({"Lewis Hamilton": career("Lewis Hamilton"),
                    "Max Verstappen": career("Max Verstappen")})
print(h2h)

# Same-race meetings: who actually finished ahead when both raced?
ham = fa[fa["driver"] == "Lewis Hamilton"][["raceId", "positionOrder"]].rename(columns={"positionOrder": "ham"})
ver = fa[fa["driver"] == "Max Verstappen"][["raceId", "positionOrder"]].rename(columns={"positionOrder": "ver"})
m = ham.merge(ver, on="raceId")
print(f"\nRaces where both started: {len(m)}")
print(f"Hamilton finished ahead: {(m['ham'] < m['ver']).sum()}   |   "
      f"Verstappen finished ahead: {(m['ver'] < m['ham']).sum()}")

              Lewis Hamilton  Max Verstappen
entries              387.000         240.000
wins                 106.000          71.000
podiums              206.000         128.000
poles                104.000          48.000
fastest_laps          68.000          37.000
points              5057.500        3350.500
win_rate               0.274           0.296
avg_start              4.600           4.930
avg_finish             3.970           3.750

Races where both started: 239
Hamilton finished ahead: 134   |   Verstappen finished ahead: 105


## 12) Do Asian races produce more DNFs than European races?

A **DNF** ("did not finish") = a car that wasn't classified at the end, i.e.
`position == "\N"` → `finish_n` is NaN (retirements, crashes, mechanical).

The race's **country** comes from `circuits.csv` (joined via `races.circuitId`),
which we bucket into **Asia** / **Europe**. Azerbaijan and Turkey are
transcontinental — their circuits sit on the Asian side, so they go in Asia;
Russia (Sochi) goes in Europe. Other continents are left out of this comparison.

⚠️ **Watch for the era trap:** European races span the unreliable 1950s–90s,
while almost every Asian venue only appears from ~2004 onward. So a raw Asia-vs-
Europe number mostly measures *which decades* each set of races covers, not the
geography — we control for that by also comparing within the same years.

In [ ]:
# Tag every result with the race's continent and whether it was a DNF
asia = {"Bahrain", "China", "India", "Japan", "Korea", "Malaysia", "Qatar",
        "Saudi Arabia", "Singapore", "UAE", "Azerbaijan", "Turkey"}
europe = {"Austria", "Belgium", "France", "Germany", "Hungary", "Italy", "Monaco",
          "Netherlands", "Portugal", "Russia", "Spain", "Sweden", "Switzerland", "UK"}

def continent(c):
    if c in asia:   return "Asia"
    if c in europe: return "Europe"
    return "Other"

race_loc = races.merge(circuits[["circuitId", "country"]], on="circuitId", how="left")
race_loc["continent"] = race_loc["country"].map(continent)

dnf = results.merge(race_loc[["raceId", "country", "continent", "year"]], on="raceId", how="left")
dnf["dnf"] = pd.to_numeric(dnf["position"], errors="coerce").isna()   # not classified = DNF

ae = dnf[dnf["continent"].isin(["Asia", "Europe"])]

# (a) raw, all years
raw = ae.groupby("continent")["dnf"].agg(entries="size", dnf_rate="mean")
raw["dnf_rate"] = (raw["dnf_rate"] * 100).round(1)
print("RAW (all years):")
print(raw)

# (b) same-era comparison to remove the reliability/era confounder
for since in (2004, 2014):
    e = ae[ae["year"] >= since].groupby("continent")["dnf"].agg(entries="size", dnf_rate="mean")
    e["dnf_rate"] = (e["dnf_rate"] * 100).round(1)
    print(f"\nSince {since}:")
    print(e)

In [ ]:
# Per-country DNF rate (so you can see which venues drive the averages)
per_country = (ae.groupby(["continent", "country"])["dnf"]
                 .agg(entries="size", dnf_rate="mean"))
per_country["dnf_rate"] = (per_country["dnf_rate"] * 100).round(1)
print("DNF rate by country (Asia vs Europe):")
print(per_country.sort_values(["continent", "dnf_rate"], ascending=[True, False]))